In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate torch evaluate scikit-learn huggingface_hub



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login
import os

# Авторизация в Hugging Face (если модель gated)
# Получите токен на https://huggingface.co/settings/tokens
login(token="hf_ВАШ_ТОКЕН_ЗДЕСЬ")  # Замените на ваш токен


In [ ]:
squad = load_dataset("squad_v2")
print(f"Train size: {len(squad['train'])}")
print(f"Validation size: {len(squad['validation'])}")

# Для ускорения обучения можно взять подмножество данных
# Раскомментируйте следующие строки, если хотите обучать на меньшем объеме
# squad["train"] = squad["train"].select(range(10000))
# squad["validation"] = squad["validation"].select(range(1000))

# Разбиение train на train/val (90/10)
train_data = squad["train"].shuffle(seed=42)
train_val_split = train_data.train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train": train_val_split["train"],
    "validation": train_val_split["test"]
})

print(f"New train size: {len(dataset['train'])}")
print(f"New validation size: {len(dataset['validation'])}")

In [ ]:
def format_prompt(context, question, answer=None):
    """Форматирование промпта для QA задачи"""
    prompt = f"""Context: {context}

Question: {question}

Answer: """
    
    if answer is not None:
        # Для обучения: добавляем ответ
        if isinstance(answer, dict):
            # SQuAD формат ответа
            answer_text = answer["text"][0] if answer["text"] else "No answer"
        else:
            answer_text = answer
        prompt += answer_text
    
    return prompt

def preprocess_function(examples, tokenizer):
    """Токенизация датасета"""
    prompts = []
    answers = []
    
    for context, question, answer in zip(examples["context"], examples["question"], examples["answers"]):
        # Формируем промпт без ответа для input
        prompt = format_prompt(context, question)
        prompts.append(prompt)
        
        # Получаем текст ответа для labels
        if answer["text"]:
            answers.append(answer["text"][0])
        else:
            answers.append("No answer")
    
    # Токенизация промптов
    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )
    
    # Токенизация ответов для labels
    labels = tokenizer(
        answers,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors=None
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
torch.cuda.empty_cache()

# Имя модели
model_name = "meta-llama/Llama-3.2-1B"
# Альтернативная модель, если нет доступа к Llama:
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {model_name}")

# Конфигурация 4-bit quantization с поддержкой CPU offload
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True  # Включаем offload на CPU при нехватке памяти
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Проверка доступной памяти GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"GPU memory free: {torch.cuda.memory_reserved(0) / 1024**3:.1f} GB")

# Определение device_map в зависимости от доступной памяти
try:
    # Пробуем загрузить всё на GPU
    device_map = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map=device_map,
        trust_remote_code=True,
        torch_dtype=torch.float16
    )
    print("Model loaded entirely on GPU")
except Exception as e:
    print(f"Failed to load on GPU: {e}")
    print("Falling back to auto device_map with CPU offload")
    # Если не хватает памяти, используем auto с CPU offload
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        max_memory={0: "6GB", "cpu": "20GB"}  # Настройте под вашу систему
    )
    print("Model loaded with CPU offload")

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

# Включение градиентного checkpointing для экономии памяти
model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded successfully!")


In [ ]:
lora_config = LoraConfig(
    r=8,  # Уменьшенный rank для экономии памяти (было 16)
    lora_alpha=16,  # Уменьшенный scaling factor (было 32)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
print("Trainable parameters:")
model.print_trainable_parameters()

In [ ]:
def tokenize_dataset(examples):
    return preprocess_function(examples, tokenizer)

tokenized_dataset = dataset.map(
    tokenize_dataset,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset"
)

# Установка формата для PyTorch
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenized train size: {len(tokenized_dataset['train'])}")
print(f"Tokenized validation size: {len(tokenized_dataset['validation'])}")

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Аргументы обучения
training_args = TrainingArguments(
    output_dir="./squad-llama-lora",
    num_train_epochs=2,  # Уменьшено с 3 для ускорения
    per_device_train_batch_size=1,  # Уменьшено с 4 для экономии памяти
    per_device_eval_batch_size=1,   # Уменьшено с 4
    gradient_accumulation_steps=8,  # Увеличено для компенсации малого batch size
    warmup_ratio=0.1,
    learning_rate=2e-4,
    fp16=True,  # Используем fp16 для ускорения
    logging_steps=10,  # Чаще логируем
    evaluation_strategy="steps",
    eval_steps=50,  # Чаще оцениваем
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",  # Отключаем wandb/tensorboard
    push_to_hub=False,
    gradient_checkpointing=True,  # Экономия памяти
    optim="paged_adamw_8bit",  # Оптимизированный оптимизатор
    dataloader_num_workers=0,  # Избегаем проблем с многопоточностью
)

# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("Trainer initialized!")
print(f"Training batch size: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Number of training steps: {len(tokenized_dataset['train']) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")


In [ ]:
print("=" * 50)
print("Starting training...")
print("=" * 50)

try:
    trainer.train()
    print("Training completed successfully!")
except Exception as e:
    print(f"Error during training: {e}")
    raise

In [ ]:
output_dir = "./lora-squad-model"
print(f"Saving model to {output_dir}")

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved successfully!")
print(f"Model size: {sum(os.path.getsize(os.path.join(output_dir, f)) for f in os.listdir(output_dir)) / 1024**2:.2f} MB")

In [ ]:
def test_model(context, question):
    """Тестирование сохраненной модели"""
    prompt = f"""Context: {context}

Question: {question}

Answer: """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = generated_text.split("Answer:")[-1].strip().split('\n')[0]
    
    return answer

# Тестовые примеры
test_examples = [
    {
        "context": "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.",
        "question": "Who designed the Eiffel Tower?"
    },
    {
        "context": "Python is an interpreted, high-level and general-purpose programming language. Created by Guido van Rossum and first released in 1991.",
        "question": "Who created Python?"
    }
]

print("\n" + "=" * 50)
print("Testing the fine-tuned model:")
print("=" * 50)

for i, example in enumerate(test_examples, 1):
    print(f"\nExample {i}:")
    print(f"Question: {example['question']}")
    answer = test_model(example['context'], example['question'])
    print(f"Answer: {answer}")

print("\nDone!")